# Write Zou 2022 longitudinal OD600 and OD445 as linked ATST

Run from this folder. One OD600 readout is made per selected worksheet; Figures 4e and 4f also have separate OD445 readouts for the DAAO reaction absorbance columns. Panels sharing a worksheet are aligned on the union of their reported times.


In [1]:
from pathlib import Path
import datetime
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    ReadoutIds,
    Study,
)

root = Path.cwd()
source_name = "41467_2022_31934_MOESM7_ESM.xlsx"
workbook = load_workbook(
    root / "sources" / source_name, read_only=False, data_only=True
)
file_info = FileMetadata(
    data={
        "file_name": "atst_zou_2022.atst.txt",
        "format": "ATST",
        "format_version": "0.1",
        "created_on": datetime.date.today().isoformat(),
        "field_delimiter": "TAB",
        "encoding": "UTF-8",
    }
)
study = Study(
    data={
        "title": "Systematic strategies for developing phage resistant Escherichia coli strains",
        "study_id": "doi:10.1038/s41467-022-31934-9",
        "authors": "Xuan Zou et al.",
        "publication_doi": "10.1038/s41467-022-31934-9",
        "publication_date": "2022",
        "description": "Source-data OD600 growth and fermentation series, plus separate 445-nm absorbance series for DAAO assay.",
    }
)
entities = Entities(
    tables={
        name: EntityTable(
            name="ENTITIES",
            table_name=name,
            pk=pk,
            data=pd.read_csv(
                root / "entities_definitions" / f"{name}.csv",
                dtype=str,
                keep_default_na=False,
            ),
        )
        for name, pk in [
            ("ISOLATES", "ISOLATE_ID"),
            ("PHAGES", "PHAGE_ID"),
            ("MEDIA", "MEDIUM_ID"),
        ]
    }
)


def curve_id(index):
    letters = ""
    while index:
        index, remainder = divmod(index - 1, 26)
        letters = chr(65 + remainder) + letters
    return "curve_" + letters + ("_238" if letters == "ID" else "")


def isolate_id(label):
    text = label.lower().replace(" ", "")
    if "w3110" in text:
        base = "W3110"
    elif "de3" in text:
        base = "MG1655DE3"
    else:
        base = "MG1655"
    if "pwhu3640" in text or "pt+" in text:
        return "MG1655_pWHU3640"
    if "sk+" in text or "emptyvector" in text:
        return "MG1655_pSK_empty"
    if "-pt" in text:
        return "MG1655_PT"
    return base + ("_EPR" if "epr" in text else "_WT")


def phage_id(label, panel):
    if panel == "Figure 3c":
        return "phage_cocktail"
    if "no phage" in label.lower() or panel in {
        "Figure 1b",
        "Figure 3b",
        "Supplementary figure 3",
    }:
        return ""
    for name in ["JMPW2", "lambda", "EEP", "T1", "T4", "T5", "T7"]:
        if re.search(r"(?<![A-Za-z0-9])" + name + r"(?![A-Za-z0-9])", label, re.I):
            return name
    return ""


def make_readout(readout_id, sheet_name, kind, curves, blocks):
    times = sorted({t for row, values in curves for t in values})
    readings = pd.DataFrame({"Time": times})
    layout_rows = []
    for index, (row, values) in enumerate(curves, 1):
        cid = curve_id(index)
        readings[cid] = [values.get(t) for t in times]
        layout_rows.append({"curve_id": cid, **row})
    fed = kind in {"fermentation_od600", "absorbance_445"}
    assay_data = {
        "readout_type": "absorbance",
        "readout_unit": "od445" if kind == "absorbance_445" else "od600",
        "time_unit": "h",
        "medium_id": "fed_batch" if fed else "LB",
        "culture_type": "fed-batch bioreactor"
        if fed
        else "mixed microplate and shake-flask"
        if readout_id == "Fig_3abc_OD600"
        else "planktonic microplate",
        "conditions": "445-nm absorbance of the DAAO activity reaction mixture; distinct from culture OD600"
        if kind == "absorbance_445"
        else "Fed-batch bioreactor cell density, duplicate profiles; phage cocktail added at fermentation start"
        if fed
        else "Phage infection growth curves; 200 µL in 96-well plate at 37 °C with agitation"
        if readout_id != "Fig_3abc_OD600"
        else "Figure 3a microplate phage curves; Figures 3b/c shake-flask uninfected/cocktail curves",
        "source_reference": "doi:10.1038/s41467-022-31934-9",
    }
    if kind == "fermentation_od600":
        assay_data["bioreactor"] = "5 L BIOTECH2002"
    elif not fed:
        assay_data["instrument"] = "Multiskan GO microplate spectrophotometer (Figure 3b/c flask OD600 instrument: Ultrospec 3000)"
    metadata = Metadata(
        data={
            "source_file": "read_shared_data/sources/" + source_name,
            "source_blocks": " | ".join(blocks),
            "conversion": "Source numeric readings and time values retained; distinct source panels aligned by union of times with empty cells where a curve has no measurement",
        }
    )
    return ATSTFile(
        file_info=file_info,
        study=study,
        metadata=metadata,
        assay=Assay(data=assay_data),
        entities=entities,
        layout=Layout(data=pd.DataFrame(layout_rows)),
        readings=Readings(data=readings),
    )


readouts = {}
small_sheets = [
    ("Fig_1bc_OD600", "Fig. 1b-1c ", 1),
    ("Fig_3abc_OD600", "Fig. 3a-c", 1),
    ("SFig_3_OD600", "SFig. 3", 1),
    ("SFig_6_OD600", "SFig. 6", 2),
]
for readout_id, sheet_name, time_col in small_sheets:
    sheet = workbook[sheet_name]
    curves = []
    blocks = []
    panel = ""
    for r in range(1, sheet.max_row + 1):
        first = sheet.cell(r, 1).value
        if isinstance(first, str) and (
            first.startswith("Figure ") or first.startswith("Supplementary figure")
        ):
            panel = first
        if sheet.cell(r, time_col).value != "Time(h)":
            continue
        label_row = (
            r
            if any(
                isinstance(sheet.cell(r, c).value, str)
                and "Replicate" not in sheet.cell(r, c).value
                for c in range(time_col + 1, sheet.max_column + 1)
            )
            else r - 1
        )
        data_rows = []
        k = r + 1
        if (
            k <= sheet.max_row
            and not isinstance(sheet.cell(k, time_col).value, (int, float))
            and str(sheet.cell(k, time_col + 1).value).startswith("Replicate")
        ):
            k += 1
        while k <= sheet.max_row and isinstance(
            sheet.cell(k, time_col).value, (int, float)
        ):
            data_rows.append(k)
            k += 1
        if not data_rows:
            continue
        starts = [
            c
            for c in range(time_col + 1, sheet.max_column + 1)
            if isinstance(sheet.cell(label_row, c).value, str)
            and sheet.cell(label_row, c).value.strip()
            and "Replicate" not in sheet.cell(label_row, c).value
        ]
        blocks.append(
            f"{panel}: {get_column_letter(time_col)}{data_rows[0]}:{get_column_letter(time_col)}{data_rows[-1]}; labels row {label_row}; {len(starts)} groups"
        )
        for col in starts:
            label = sheet.cell(label_row, col).value.strip()
            pid = phage_id(label, panel)
            isolate = isolate_id(label)
            moi_match = re.search(r"MOI\s*=\s*([0-9.]+)", label, re.I)
            assay_mode = (
                "shake_flask" if panel in {"Figure 3b", "Figure 3c"} else "microplate"
            )
            for replicate in range(1, 4):
                source_col = col + replicate - 1
                values = {
                    float(sheet.cell(i, time_col).value): sheet.cell(
                        i, source_col
                    ).value
                    for i in data_rows
                }
                assert all(
                    v is None or isinstance(v, (int, float)) for v in values.values()
                ), (sheet_name, label, source_col)
                curves.append(
                    (
                        {
                            "type": "phage_exposed" if pid else "uninfected_control",
                            "isolate_id": isolate,
                            "phage_id": pid,
                            "moi": moi_match.group(1) if moi_match else "",
                            "replicate": str(replicate),
                            "assay_mode": assay_mode,
                        },
                        values,
                    )
                )
    readouts[readout_id] = make_readout(readout_id, sheet_name, "od600", curves, blocks)

for figure in ["4e", "4f", "4g"]:
    sheet_name = "Fig. " + figure
    sheet = workbook[sheet_name]
    time_col = 7 if figure == "4g" else 8
    curves = []
    blocks = []
    condition = ""
    for r in range(1, sheet.max_row + 1):
        value = sheet.cell(r, time_col).value
        if isinstance(value, str) and "phage cocktail" in value:
            condition = value
        if value != "Time (h)":
            continue
        data_rows = []
        k = r + 1
        while k <= sheet.max_row and isinstance(
            sheet.cell(k, time_col).value, (int, float)
        ):
            data_rows.append(k)
            k += 1
        if not data_rows:
            continue
        blocks.append(
            f"{condition}: {get_column_letter(time_col)}{data_rows[0]}:{get_column_letter(time_col + 2)}{data_rows[-1]}"
        )
        for replicate in range(1, 3):
            col = time_col + replicate
            values = {
                float(sheet.cell(i, time_col).value): sheet.cell(i, col).value
                for i in data_rows
            }
            assert all(isinstance(v, (int, float)) for v in values.values()), (
                sheet_name,
                r,
                col,
            )
            infected = (
                "with phage cocktail" in condition
                and "without phage cocktail" not in condition
            )
            curves.append(
                (
                    {
                        "type": "phage_exposed" if infected else "uninfected_control",
                        "isolate_id": isolate_id(condition),
                        "phage_id": "phage_cocktail" if infected else "",
                        "moi": "",
                        "replicate": str(replicate),
                        "assay_mode": "fed_batch",
                    },
                    values,
                )
            )
    readout_id = "Fig_" + figure + "_OD600"
    readouts[readout_id] = make_readout(
        readout_id, sheet_name, "fermentation_od600", curves, blocks
    )

for figure in ["4e", "4f"]:
    sheet_name = "Fig. " + figure
    sheet = workbook[sheet_name]
    curves = []
    blocks = []
    condition = ""
    for r in range(1, sheet.max_row + 1):
        first = sheet.cell(r, 1).value
        if (
            isinstance(first, str)
            and "DAAO enzyme" in first
            and "phage cocktail" in first
        ):
            condition = first
        if first != "Time (h)" or sheet.cell(r, 3).value != "Absorption (445nm)":
            continue
        data_rows = []
        k = r + 1
        while (
            k <= sheet.max_row
            and sheet.cell(k, 1).value is None
            and sheet.cell(k, 2).value is None
        ):
            k += 1
        while k <= sheet.max_row and (
            isinstance(sheet.cell(k, 1).value, (int, float))
            or (
                sheet.cell(k, 1).value is None
                and isinstance(sheet.cell(k, 2).value, str)
                and "Replicate" in sheet.cell(k, 2).value
            )
        ):
            data_rows.append(k)
            k += 1
        if not data_rows:
            continue
        blocks.append(f"{condition}: A{data_rows[0]}:C{data_rows[-1]}")
        for replicate in range(1, 3):
            values = {}
            current_time = None
            for i in data_rows:
                if isinstance(sheet.cell(i, 1).value, (int, float)):
                    current_time = float(sheet.cell(i, 1).value)
                if (
                    sheet.cell(i, 2).value
                    == f"{'EPR' if 'EPR' in condition else 'WT'}-Replicate #{replicate}"
                ):
                    values[current_time] = sheet.cell(i, 3).value
            assert values and all(
                isinstance(v, (int, float)) for v in values.values()
            ), (sheet_name, r, replicate)
            infected = (
                "with phage cocktail" in condition
                and "without phage cocktail" not in condition
            )
            curves.append(
                (
                    {
                        "type": "phage_exposed" if infected else "uninfected_control",
                        "isolate_id": isolate_id(condition),
                        "phage_id": "phage_cocktail" if infected else "",
                        "moi": "",
                        "replicate": str(replicate),
                        "assay_mode": "DAAO_reaction",
                    },
                    values,
                )
            )
    readout_id = "Fig_" + figure + "_OD445"
    readouts[readout_id] = make_readout(
        readout_id, sheet_name, "absorbance_445", curves, blocks
    )

readout_ids = ReadoutIds(data=pd.DataFrame({"readout_id": list(readouts)}))
for item in readouts.values():
    item.readout_ids = readout_ids
bundle = MultiReadoutATST(file_info, study, readout_ids, readouts, entities)
validate_atst(bundle)
display(
    pd.DataFrame(
        [
            {
                "readout_id": k,
                "curves": len(v.layout.data),
                "time_points": len(v.readings.data),
            }
            for k, v in readouts.items()
        ]
    )
)


       readout_id  curves  time_points
0   Fig_1bc_OD600     132           25
1  Fig_3abc_OD600     162           11
2    SFig_3_OD600      12           43
3    SFig_6_OD600     315           11
4    Fig_4e_OD600       8           23
5    Fig_4f_OD600       8           23
6    Fig_4g_OD600       8           23
7    Fig_4e_OD445       8           22
8    Fig_4f_OD445       8           22


In [2]:
output = write_multi_readout_atst(
    bundle, root.parent / "atst_zou_2022", linked_files=True
)
print(output)


/Users/juanmantilla/Documents/TKI/ATST/examples/Public data /Zou 2022/atst_zou_2022/atst_zou_2022.atst.txt
